In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ast

In [24]:
df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\notebooks\final_df.csv')

In [26]:
df = df.drop(columns=['keywords_comments'])
df.head()

,post_id,postUser,timestamp,combined_tokens,likesCount,commentsCount,score_segtiment,keywords_posts
0,1,maryleest,2023-06-02 16:34:43+00:00,"['cannes', '2023', 'kilianparis', 'kiliancanne...",32070,142,0.403500,"['kilian', 'kilianparis', 'white', 'wearing', ..."
1,2,tinaabeysekara,2023-01-01 07:49:16+00:00,"['clock', 'struck', 'midnight', 'ring', '2022'...",6565,86,0.316667,"['clock', 'thank', '2022', 'midnight', 'face',..."
2,3,maryleest,2023-05-29 18:57:51+00:00,"['famous', 'stairs', 'photo', 'gustave_durin',...",28936,123,0.400000,"['famous', 'jewelry', 'stairs', 'photo', 'marm..."
3,4,stephaniebroek,2023-03-07 19:30:42+00:00,"['visualized', 'moment', 'many', 'times', 'fir...",4764,215,0.168889,"['fashion', 'chanel', 'chanelofficial', 'edito..."
4,5,maryleest,2023-05-23 21:11:12+00:00,"['got', 'witness', 'historical', 'moment', 'ci...",13379,123,0.401000,"['historical', 'witness', 'got', 'moment', 'ca..."


In [30]:
comments_df = pd.read_csv(r'C:\Users\ADMIN\Desktop\ITDSIU21099_HoangVanManh\Fashion-Marketing-Automation-Solutions\notebooks\new_comments.csv')
comments_df = comments_df.drop(columns=['comments','preferences','emoji_present'])
comments_df.rename(columns={"tfidf_keywords": "keywords_comments"}, inplace=True)
comments_df.head()

,post_id,commentUser,timestamp,clean_comments,tokens,polarity,sentiment,keywords_comments
0,1,wilsonjunior5055,2023-06-05 12:27:24+00:00,i have a proposal for u send me a dm please,"['proposal', 'u', 'send', 'dm', 'please']",0.00,neutral,"['please', 'amazing', 'beautiful', 'clapping_h..."
1,1,mariavmh,2023-06-04 16:46:40+00:00,woaaaah,['woaaaah'],0.00,neutral,"['amazing', 'beautiful', 'clapping_hands', 'de..."
2,1,mariavmh,2023-06-04 16:46:34+00:00,smiling_face_with_hearteyessmiling_face_with_h...,['smiling_face_with_hearteyessmiling_face_with...,0.96,positive,"['amazing', 'beautiful', 'clapping_hands', 'de..."
3,1,stylemesoftly_,2023-06-04 15:44:54+00:00,red_heartred_heartred_heart,['red_heartred_heartred_heart'],0.24,positive,"['red_heartred_heartred_heart', 'amazing', 'be..."
4,1,jillviv,2023-06-04 15:06:03+00:00,smiling_face_with_hearteyessmiling_face_with_h...,['smiling_face_with_hearteyessmiling_face_with...,0.70,positive,['smiling_face_with_hearteyessmiling_face_with...


In [38]:
user_profile = comments_df.groupby("commentUser").agg({
    "post_id": "count", # Số lượng bình luận
    "polarity": "mean",  # Điểm cảm xúc trung bình (avg_sentiment_score)
    "keywords_comments": lambda x: " ".join(x),  # Gộp từ khóa từ bình luận
}).reset_index()

In [39]:
user_profile.head()

,commentUser,post_id,polarity,keywords_comments
0,021_alaniss,1,0.80,"['fire', 'amazing', 'beautiful', 'clapping_han..."
1,100ycientas,1,0.24,"['amazing', 'beautiful', 'clapping_hands', 'de..."
2,1072.official,1,0.52,"['amazing', 'smiling_face_with_hearteyes', 'be..."
3,12maaria34,1,0.00,"['amazing', 'beautiful', 'clapping_hands', 'de..."
4,130wlifestyle,1,0.60,"['amazing', 'beautiful', 'clapping_hands', 'de..."


In [40]:
def build_interaction_sequence(df):
    # Sắp xếp theo timestamp
    df_sorted = df.sort_values('timestamp')
    # Lấy thông tin cần thiết cho mỗi event
    return df_sorted[['post_id', 'timestamp', 'polarity']].to_dict('records')

df_sequence = comments_df.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(
    columns={0: 'interaction_sequence'}
)


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_10368\3264974346.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sequence = comments_df.groupby('commentUser').apply(build_interaction_sequence).reset_index().rename(


In [41]:
df_sequence.head()

,commentUser,interaction_sequence
0,021_alaniss,"[{'post_id': 583, 'timestamp': '2023-06-02 21:..."
1,100ycientas,"[{'post_id': 319, 'timestamp': '2023-03-29 14:..."
2,1072.official,"[{'post_id': 65, 'timestamp': '2023-04-07 16:0..."
3,12maaria34,"[{'post_id': 79, 'timestamp': '2023-05-05 21:4..."
4,130wlifestyle,"[{'post_id': 272, 'timestamp': '2023-06-06 01:..."


In [43]:
df_user_profile = user_profile.merge(df_sequence, on='commentUser', how='left')
df_user_profile.head()

,commentUser,post_id,polarity,keywords_comments,interaction_sequence
0,021_alaniss,1,0.80,"['fire', 'amazing', 'beautiful', 'clapping_han...","[{'post_id': 583, 'timestamp': '2023-06-02 21:..."
1,100ycientas,1,0.24,"['amazing', 'beautiful', 'clapping_hands', 'de...","[{'post_id': 319, 'timestamp': '2023-03-29 14:..."
2,1072.official,1,0.52,"['amazing', 'smiling_face_with_hearteyes', 'be...","[{'post_id': 65, 'timestamp': '2023-04-07 16:0..."
3,12maaria34,1,0.00,"['amazing', 'beautiful', 'clapping_hands', 'de...","[{'post_id': 79, 'timestamp': '2023-05-05 21:4..."
4,130wlifestyle,1,0.60,"['amazing', 'beautiful', 'clapping_hands', 'de...","[{'post_id': 272, 'timestamp': '2023-06-06 01:..."


In [45]:
# Fill null values in polarity with 0
df_user_profile['polarity'] = df_user_profile['polarity'].fillna(0)
df_user_profile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4089 entries, 0 to 4088
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   commentUser           4089 non-null   object 
 1   post_id               4089 non-null   int64  
 2   polarity              4089 non-null   float64
 3   keywords_comments     4089 non-null   object 
 4   interaction_sequence  4089 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 159.9+ KB
